<div style="text-align: center;">
 <img src="./Image/regular_loss.png" alt="Example image">
 </div>
 <div>
 <div style="float: left; width: 50%;text-align: center;">
  <img src="./Image/L1regul.png" alt="Image 1" style="width: 50%;"/>
</div>
<div style="float: left; width: 50%;text-align: center;">
  <img src="./Image/L2_Reg.png" alt="Image 2" style="width: 50%;"/>
</div>

 </div>

## Imports & Dataset

In [2]:
import numpy as np
import nnfs
from nnfs.datasets import spiral_data

nnfs.init()

## Dense Layer (with L1/L2 Regularization)

In [3]:
class Layer_Dense:
    def __init__(self, n_inputs, n_neurons,
                 weight_regularizer_l1=0, weight_regularizer_l2=0,
                 bias_regularizer_l1=0, bias_regularizer_l2=0):

        self.weights = 0.01 * np.random.randn(n_inputs, n_neurons)
        self.biases = np.zeros((1, n_neurons))

        self.weight_regularizer_l1 = weight_regularizer_l1
        self.weight_regularizer_l2 = weight_regularizer_l2
        self.bias_regularizer_l1 = bias_regularizer_l1
        self.bias_regularizer_l2 = bias_regularizer_l2

    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.dot(inputs, self.weights) + self.biases

    def backward(self, dvalues):
        self.dweights = np.dot(self.inputs.T, dvalues)
        self.dbiases = np.sum(dvalues, axis=0, keepdims=True)

        if self.weight_regularizer_l1 > 0:
            dL1 = np.ones_like(self.weights)
            dL1[self.weights < 0] = -1
            self.dweights += self.weight_regularizer_l1 * dL1

        if self.weight_regularizer_l2 > 0:
            self.dweights += 2 * self.weight_regularizer_l2 * self.weights

        if self.bias_regularizer_l1 > 0:
            dL1 = np.ones_like(self.biases)
            dL1[self.biases < 0] = -1
            self.dbiases += self.bias_regularizer_l1 * dL1

        if self.bias_regularizer_l2 > 0:
            self.dbiases += 2 * self.bias_regularizer_l2 * self.biases

        self.dinputs = np.dot(dvalues, self.weights.T)

## Activation Functions

In [4]:
class Activation_ReLU:
    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.maximum(0, inputs)

    def backward(self, dvalues):
        self.dinputs = dvalues.copy()
        self.dinputs[self.inputs <= 0] = 0

In [5]:
class Activation_Softmax:
    def forward(self, inputs):
        self.inputs = inputs
        exp_values = np.exp(inputs - np.max(inputs, axis=1, keepdims=True))
        self.output = exp_values / np.sum(exp_values, axis=1, keepdims=True)

    def backward(self, dvalues):
        self.dinputs = np.empty_like(dvalues)

        for i, (output, dval) in enumerate(zip(self.output, dvalues)):
            output = output.reshape(-1, 1)
            jacobian = np.diagflat(output) - np.dot(output, output.T)
            self.dinputs[i] = np.dot(jacobian, dval)

## Loss Functions

In [6]:
class Loss:
    def regularization_loss(self, layer):
        loss = 0

        if layer.weight_regularizer_l1 > 0:
            loss += layer.weight_regularizer_l1 * np.sum(np.abs(layer.weights))

        if layer.weight_regularizer_l2 > 0:
            loss += layer.weight_regularizer_l2 * np.sum(layer.weights * layer.weights)

        if layer.bias_regularizer_l1 > 0:
            loss += layer.bias_regularizer_l1 * np.sum(np.abs(layer.biases))

        if layer.bias_regularizer_l2 > 0:
            loss += layer.bias_regularizer_l2 * np.sum(layer.biases * layer.biases)

        return loss

    def calculate(self, output, y):
        return np.mean(self.forward(output, y))

In [7]:
class Loss_CategoricalCrossentropy(Loss):
    def forward(self, y_pred, y_true):
        samples = len(y_pred)
        y_pred = np.clip(y_pred, 1e-7, 1 - 1e-7)

        if len(y_true.shape) == 1:
            correct = y_pred[range(samples), y_true]
        else:
            correct = np.sum(y_pred * y_true, axis=1)

        return -np.log(correct)

    def backward(self, dvalues, y_true):
        samples = len(dvalues)
        labels = len(dvalues[0])

        if len(y_true.shape) == 1:
            y_true = np.eye(labels)[y_true]

        self.dinputs = -y_true / dvalues
        self.dinputs = self.dinputs / samples

## Combined Softmax + Loss

In [8]:
class Activation_Softmax_Loss_CategoricalCrossentropy:
    def __init__(self):
        self.activation = Activation_Softmax()
        self.loss = Loss_CategoricalCrossentropy()

    def forward(self, inputs, y):
        self.activation.forward(inputs)
        self.output = self.activation.output
        return self.loss.calculate(self.output, y)

    def backward(self, dvalues, y):
        samples = len(dvalues)

        if len(y.shape) == 2:
            y = np.argmax(y, axis=1)

        self.dinputs = dvalues.copy()
        self.dinputs[range(samples), y] -= 1
        self.dinputs = self.dinputs / samples

## Optimizers

In [9]:
class Optimizer_SGD:
    def __init__(self, learning_rate=1., decay=0., momentum=0.):
        self.lr = learning_rate
        self.current_lr = learning_rate
        self.decay = decay
        self.iterations = 0
        self.momentum = momentum

    def pre_update_params(self):
        if self.decay:
            self.current_lr = self.lr * (1. / (1. + self.decay * self.iterations))

    def update_params(self, layer):
        if self.momentum:
            if not hasattr(layer, 'weight_momentums'):
                layer.weight_momentums = np.zeros_like(layer.weights)
                layer.bias_momentums = np.zeros_like(layer.biases)

            w_updates = self.momentum * layer.weight_momentums - self.current_lr * layer.dweights
            b_updates = self.momentum * layer.bias_momentums - self.current_lr * layer.dbiases

            layer.weight_momentums = w_updates
            layer.bias_momentums = b_updates
        else:
            w_updates = -self.current_lr * layer.dweights
            b_updates = -self.current_lr * layer.dbiases

        layer.weights += w_updates
        layer.biases += b_updates

    def post_update_params(self):
        self.iterations += 1

In [10]:
class Optimizer_Adam:
    def __init__(self, learning_rate=0.001, decay=0., epsilon=1e-7,
                 beta_1=0.9, beta_2=0.999):

        self.lr = learning_rate
        self.current_lr = learning_rate
        self.decay = decay
        self.iterations = 0
        self.epsilon = epsilon
        self.beta1 = beta_1
        self.beta2 = beta_2

    def pre_update_params(self):
        if self.decay:
            self.current_lr = self.lr * (1. / (1. + self.decay * self.iterations))

    def update_params(self, layer):
        if not hasattr(layer, 'weight_cache'):
            layer.weight_momentums = np.zeros_like(layer.weights)
            layer.weight_cache = np.zeros_like(layer.weights)
            layer.bias_momentums = np.zeros_like(layer.biases)
            layer.bias_cache = np.zeros_like(layer.biases)

        layer.weight_momentums = self.beta1 * layer.weight_momentums + (1 - self.beta1) * layer.dweights
        layer.bias_momentums = self.beta1 * layer.bias_momentums + (1 - self.beta1) * layer.dbiases

        wm_corr = layer.weight_momentums / (1 - self.beta1 ** (self.iterations + 1))
        bm_corr = layer.bias_momentums / (1 - self.beta1 ** (self.iterations + 1))

        layer.weight_cache = self.beta2 * layer.weight_cache + (1 - self.beta2) * layer.dweights**2
        layer.bias_cache = self.beta2 * layer.bias_cache + (1 - self.beta2) * layer.dbiases**2

        wc_corr = layer.weight_cache / (1 - self.beta2 ** (self.iterations + 1))
        bc_corr = layer.bias_cache / (1 - self.beta2 ** (self.iterations + 1))

        layer.weights += -self.current_lr * wm_corr / (np.sqrt(wc_corr) + self.epsilon)
        layer.biases += -self.current_lr * bm_corr / (np.sqrt(bc_corr) + self.epsilon)

    def post_update_params(self):
        self.iterations += 1

## Model Setup

In [11]:
X, y = spiral_data(samples=100, classes=3)

dense1 = Layer_Dense(2, 64, weight_regularizer_l2=5e-4,
                     bias_regularizer_l2=5e-4)
activation1 = Activation_ReLU()

dense2 = Layer_Dense(64, 3)

loss_activation = Activation_Softmax_Loss_CategoricalCrossentropy()

optimizer = Optimizer_Adam(learning_rate=0.02, decay=5e-7)

In [12]:
import tqdm

## Training Loop

In [13]:
for epoch in range(10001):

    dense1.forward(X)
    activation1.forward(dense1.output)
    dense2.forward(activation1.output)

    data_loss = loss_activation.forward(dense2.output, y)

    reg_loss = (
        loss_activation.loss.regularization_loss(dense1) +
        loss_activation.loss.regularization_loss(dense2)
    )

    loss = data_loss + reg_loss

    predictions = np.argmax(loss_activation.output, axis=1)
    if len(y.shape) == 2:
        y = np.argmax(y, axis=1)

    accuracy = np.mean(predictions == y)

    if not epoch % 100:
        print(epoch, accuracy, loss)

    loss_activation.backward(loss_activation.output, y)
    dense2.backward(loss_activation.dinputs)
    activation1.backward(dense2.dinputs)
    dense1.backward(activation1.dinputs)

    optimizer.pre_update_params()
    optimizer.update_params(dense1)
    optimizer.update_params(dense2)
    optimizer.post_update_params()

0 0.36 1.0986004566168412
100 0.69 0.8449501132965088
200 0.7733333333333333 0.6895837779045105
300 0.8166666666666667 0.6171496715545655
400 0.8233333333333334 0.5673076455593109
500 0.83 0.5345990812778473
600 0.8433333333333334 0.50355228805542
700 0.8566666666666667 0.48096590924263
800 0.8566666666666667 0.4617595686912537
900 0.8733333333333333 0.44801131939888
1000 0.87 0.43366239857673644
1100 0.88 0.42237355184555053
1200 0.8833333333333333 0.41250973773002625
1300 0.8866666666666667 0.40342678570747376
1400 0.89 0.3958400321006775
1500 0.8933333333333333 0.3887316761016846
1600 0.9 0.38123025846481323
1700 0.8966666666666666 0.37687057685852055
1800 0.9 0.36962746262550356
1900 0.9 0.36404252648353574
2000 0.9033333333333333 0.36006971764564516
2100 0.9066666666666666 0.355933783531189
2200 0.9 0.3525214459896088
2300 0.9033333333333333 0.3487828280925751
2400 0.9033333333333333 0.3435909652709961
2500 0.92 0.3378697421550751
2600 0.8933333333333333 0.34290600490570067
2700 0

## Validation

In [14]:
X_test, y_test = spiral_data(samples=100, classes=3)

dense1.forward(X_test)
activation1.forward(dense1.output)
dense2.forward(activation1.output)

loss = loss_activation.forward(dense2.output, y_test)

predictions = np.argmax(loss_activation.output, axis=1)

if len(y_test.shape) == 2:
    y_test = np.argmax(y_test, axis=1)

accuracy = np.mean(predictions == y_test)

print("validation accuracy:", accuracy," validation Loss:", loss)

validation accuracy: 0.84  validation Loss: 0.5246933
